# Limits and continuity with **Sympy**

The **Sympy** module can be used to obtain the singularities of a function, its domain, and its range, among other things.
As we will see in this tutorial, we can also compute limits of one-variable functions and check whether a function is continuous at a point.


Let us remember that the first thing we must do is import the **Sympy** module for the rest of the tutorial:


In [2]:
import sympy as sp

## Expressions and functions in **Sympy**
Until now we have used mathematical expressions stored in objects from the **Sympy** module. However, we have not yet used functions. To see the differences between an expression and its associated function, let us show how to evaluate functions and expressions in the following example:


In [3]:
x = sp.symbols('x', real=True) # define the symbolic variable x
f_expr = x*sp.cos(4*x) # This is an expression
display(f_expr)
print('Value of f_expr(2)=',f_expr.subs({x:2})) # Evaluate the expression $f$ with "subs"

y = sp.symbols('y', real=True)
f2_expr=sp.exp(x**3+1)
sp.solve(f2_expr-y,x)

sp.solve(sp.Abs(2*x-4)-sp.Abs(x-3))

x*cos(4*x)

Value of f_expr(2)= 2*cos(8)


[1, 7/3]

In the case of functions, evaluation is much simpler: it is done as with any other predefined function (sine, cosine, exponential, *etc.*).

To define functions in **Sympy** we must use the `sp.Lambda` function:


In [4]:
f = sp.Lambda((x),f_expr) # Create the function "f" from the expression "f_expr"
display(f)
print('Value of f(2)=',f(2)) # Evaluation of the function
print(f(x)==f_expr)

Lambda(x, x*cos(4*x))

Value of f(2)= 2*cos(8)
True


## Piecewise-defined functions

Functions can also be defined **piecewise**, taking into account different expressions that will be evaluated when certain conditions are satisfied. The **Sympy** module evaluates each of the tuples passed as arguments (from left to right) and selects the first expression whose condition is satisfied. The syntax is the following:


In [5]:
g_expr = sp.Piecewise((1/(x), x>0), (1, True))
g = sp.Lambda((x), g_expr)
display(g(x))

Piecewise((1/x, x > 0), (1, True))

The previous notation, although very convenient, has many limitations (for example, for computing one-sided limits, which we will see a little later). In general, it is more useful to define piecewise functions using the **step function**, mathematically known as the **Heaviside** function, $\theta$, which is given by

$$
\theta(x)=
\begin{cases}
0 & \text{if } x<0,\\
1 & \text{if } x \geq 0.
\end{cases}
$$

The previous function

$$
g(x)=
\begin{cases}
\dfrac{1}{x} & \text{if } x\geq 0,\\
1 & \text{if } x<0,
\end{cases}
$$

would be written as


In [57]:
# Definition of the function
g1 = 1/x; g2 = 1
g_expr = g2 + (g1 - g2) * sp.Heaviside(x, 0)  
g = sp.Lambda(x, g_expr)
# Check the definition of the function g
display(sp.simplify(g(x).rewrite(sp.Piecewise)))

Piecewise((1, x <= 0), (1/x, True))

## Domain and range of a function

To compute the **domain** of a function, one may first determine the singularities of a given function. Then, the maximal domain of the function is all of $\mathbb{R}$ except for the singularities.


In [23]:
f=sp.Lambda(x, x/(sp.cos(x)))
display(sp.calculus.singularities(f(x), x))

Union(ImageSet(Lambda(_n, 2*_n*pi + pi/2), Integers), ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers))

To compute the **range** of a function (that is, its **image set**), we will use the function `sp.calculus.util.function_range`:


In [44]:
f=sp.Lambda(x, x/(x**2+1))
R = sp.calculus.util.function_range(f(x), x, sp.Reals)
display(R)

Interval(-1/2, 1/2)

**Exercise 5.1** 
Compute the singularities and the range of the function $f(x)=\displaystyle\frac{x+5}{x^3-2}+\frac{x^2}{x-2}$.
To do so, you must first define the associated expression and then the corresponding `Lambda` function. Next, compute its domain and its range. Finally, determine the image of the interval $[-1,1]$.


In [58]:
# WRITE YOUR CODE HERE

## Limits
The limits of one-variable expressions can be computed with the `sp.limit` function.
This function also allows us to compute one-sided limits, as shown in the following example:


In [65]:
g_expr = sp.cos(x)/(x+1)
g = sp.Lambda(x, g_expr)
display(g(x))

display(sp.limit(g(x),x,-1,dir='-')) # left-hand limit
display(sp.limit(g(x),x,-1,dir='+')) # right-hand limit

cos(x)/(x + 1)

-oo

oo

In this case, the limit $\displaystyle\lim_{x\to -1}g(x)$ does not exist because, as we have just seen, the one-sided limits do not coincide. But an incorrect use of the software package could lead us to an erroneous conclusion:


In [63]:
display(sp.limit(g(x),x,-1)) # It gives an (incorrect) result because, by default, it uses the value of the right-hand limit!

oo

Below we show how to compute one-sided limits of a piecewise-defined function.
In this case, the definition using `sp.Piecewise` will produce a runtime error (try it!), so we will have to define the function using our already much-disliked step function (Heaviside):


In [66]:
# Definition of the piecewise function (using the Heaviside function)
f1 = 1/x; f2 = 1
f_expr = f2 + (f1 - f2) * sp.Heaviside(x, 0)  
f = sp.Lambda(x, f_expr)
display(sp.simplify(f(x).rewrite(sp.Piecewise)))

display(sp.limit(f(x),x,0,dir='-')) # left-hand limit
display(sp.limit(f(x),x,0,dir='+')) # right-hand limit

Piecewise((1, x <= 0), (1/x, True))

1

oo

There is no problem if we want to compute limits at infinity, $x\to+\infty$ or $x\to-\infty$. In **Sympy**, the value $\infty$ is represented by `sp.oo`. For example:


In [67]:
display(sp.limit(sp.exp(x),x,-sp.oo))
display(sp.limit(sp.exp(x),x,sp.oo))

0

oo

**Exercise 5.2** 

Plot the following function and compute the indicated limits: 

$$
f(x)=
\begin{cases}
\cos(x) & \text{if } x<0,\\
\frac{x^2}{x+1} & \text{if } x \geq 0,
\end{cases}
$$

- $\lim_{x\to -1} f(x)$,
- $\lim_{x\to 1} f(x)$,
- $\lim_{x\to 0^{-}} f(x)$,
- $\lim_{x\to 0^{+}} f(x)$,
- $\lim_{x\to +\infty} f(x)$.


In [78]:
# WRITE YOUR CODE HERE

## Continuity

In the **Sympy** module there is a function that computes the domain of continuity (that is, the set of points at which the function is continuous): `sp.calculus.util.continuous_domain`


In [72]:
f=sp.Lambda(x, x/sp.cos(x))
I = sp.calculus.util.continuous_domain(f, x, sp.Reals)
display(I)

Complement(Reals, Union(ImageSet(Lambda(_n, 2*_n*pi + pi/2), Integers), ImageSet(Lambda(_n, 2*_n*pi + 3*pi/2), Integers)))

Moreover, to analyze the continuity of $f$ at a point $a$, it is enough to check that
$$
f(a)=\lim_{x\to a}f(x).
$$
For example:


In [77]:
# Domain of continuity of the absolute value function
f = sp.Lambda((x), sp.Abs(x))
I = sp.calculus.util.continuous_domain(f(x), x, sp.Reals)
display(I)

# Check continuity for the same function, but now defined piecewise
f1 = x; f2 = -x
f_expr = f2 + (f1-f2) * sp.Heaviside(x, 0)
f = sp.Lambda(x, f_expr)
# Check the definition of the function f
display(sp.simplify(f.rewrite(sp.Piecewise)))

print('The function f is continuous at x=0:', sp.limit(f(x),x,0)==f(0))
I = sp.calculus.util.continuous_domain(f(x), x, sp.Reals)
display(I)


Reals

Lambda(x, Piecewise((-x, x <= 0), (x, True)))

The function f is continuous at x=0: True


Reals

**Exercise 5.3** 
Analyze the continuity of the function from Exercise 5.2.


In [79]:
# WRITE YOUR CODE HERE